## Load Dataset and Basic Preparation

In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("../data/processed/reviews_clean.csv")

# Standardize column names for pipeline
df = df.rename(columns={
    "reviewerID": "user_id",
    "asin": "product_id",
    "overall": "rating",
    "reviewText": "review_text",
    "vote": "helpful_votes",
    "unixReviewTime": "review_timestamp"
})

# Ensure review text exists
df["review_text"] = df["clean_review_text"].fillna("") if "clean_review_text" in df.columns else df["review_text"].fillna("")

# Convert helpful votes
if "helpful_votes" in df.columns:
    df["helpful_votes"] = df["helpful_votes"].astype(str).str.replace(",", "")
    df["helpful_votes"] = pd.to_numeric(df["helpful_votes"], errors="coerce").fillna(0)
else:
    df["helpful_votes"] = 0

# Convert timestamp
df["review_date"] = pd.to_datetime(df["review_timestamp"], unit="s")

print("Dataset loaded:", df.shape)
print(df.columns)
df.head()

Dataset loaded: (851363, 11)
Index(['user_id', 'product_id', 'rating', 'review_text', 'summary', 'verified',
       'review_timestamp', 'reviewTime', 'helpful_votes', 'clean_review_text',
       'review_date'],
      dtype='object')


,user_id,product_id,rating,review_text,summary,verified,review_timestamp,reviewTime,helpful_votes,clean_review_text,review_date
0,A1D4G1SNUZWQOT,7106116521,5,exactly needed,perfect replacements!!,True,1413763200,2014-10-20,0.0,exactly needed,2014-10-20
1,A3DDWDH9PX2YX2,7106116521,2,agree review opening small almost bent hook ex...,"I agree with the other review, the opening is ...",True,1411862400,2014-09-28,3.0,agree review opening small almost bent hook ex...,2014-09-28
2,A2MWC41EW7XL15,7106116521,4,love going order another pack keep work someon...,My New 'Friends' !!,False,1408924800,2014-08-25,0.0,love going order another pack keep work someon...,2014-08-25
3,A2UH2QQ275NV45,7106116521,2,tiny opening,Two Stars,True,1408838400,2014-08-24,0.0,tiny opening,2014-08-24
4,A89F3LQADZBS5,7106116521,3,okay,Three Stars,False,1406419200,2014-07-27,0.0,okay,2014-07-27


## Compute Behavioral & Context Trust Signals

In [8]:
# ---------- BASIC REVIEW FEATURES ----------

df["review_length"] = df["review_text"].astype(str).apply(lambda x: len(x.split()))

# Remove extremely short reviews (noise)
df = df[df["review_length"] >= 3]


# ---------- PRODUCT CONTEXT ----------

df["product_mean_rating"] = df.groupby("product_id")["rating"].transform("mean")

df["rating_deviation"] = abs(df["rating"] - df["product_mean_rating"])

df["rating_score"] = 1 - (df["rating_deviation"] / 4)


# ---------- USER BEHAVIOR ----------

user_variance = df.groupby("user_id")["rating"].transform("var")

df["user_consistency"] = 1 / (1 + user_variance)

df["user_consistency"] = df["user_consistency"].fillna(1)


# ---------- HELPFUL VOTE SIGNAL ----------

df["helpful_ratio"] = df["helpful_votes"] / (df["helpful_votes"] + 1)


# ---------- VERIFIED PURCHASE ----------

df["verified_score"] = df["verified"].astype(int)


# ---------- SUSPICIOUS RULES ----------

# Short extreme reviews
df["rule_short_extreme"] = (
    (df["review_length"] < 15) &
    (df["rating"].isin([1,5]))
).astype(int)

# High review frequency
df["review_day"] = df["review_date"].dt.date
df["daily_count"] = df.groupby(["user_id","review_day"])["user_id"].transform("count")

df["rule_high_frequency"] = (df["daily_count"] > 3).astype(int)

# Rating deviation anomaly
df["rule_rating_deviation"] = (df["rating_deviation"] >= 3).astype(int)

# Duplicate reviews
df["rule_duplicate"] = df.duplicated(subset=["review_text"], keep=False).astype(int)

print("Feature signals created.")

Feature signals created.


## Suspicious Behavior Rules (Your Original Rules)

In [9]:
# ================================
# TRUST SCORE CONSTRUCTION
# ================================

# ----- Base Trust Score -----

df["base_trust"] = (
      0.35 * df["helpful_ratio"]
    + 0.25 * df["rating_score"]
    + 0.25 * df["user_consistency"]
    + 0.15 * df["verified_score"]
)


# ----- Suspicious Behaviour Penalty -----

df["penalty"] = (
      0.15 * df["rule_duplicate"]
    + 0.10 * df["rule_high_frequency"]
    + 0.05 * df["rule_short_extreme"]
    + 0.05 * df["rule_rating_deviation"]
)


# ----- Final Trust Score -----

df["trust_score"] = df["base_trust"] - df["penalty"]

# keep score between 0 and 1
df["trust_score"] = df["trust_score"].clip(0, 1)


# ================================
# WEAK LABEL CREATION
# ================================

# Fake review if trust score is low
df["fake_label"] = (df["trust_score"] < 0.40).astype(int)


# ================================
# CONFIDENCE LABELS
# ================================

df["label_confidence"] = np.where(
    df["trust_score"] >= 0.75, "high_real",
    np.where(df["trust_score"] <= 0.35, "high_fake", "uncertain")
)


# ================================
# DIAGNOSTICS
# ================================

print("\nTrust Score Statistics")
print(df["trust_score"].describe())

print("\nFake Label Distribution")
print(df["fake_label"].value_counts(normalize=True))

print("\nConfidence Distribution")
print(df["label_confidence"].value_counts())

print("\nSuspicious Rule Counts")
print(df[[
    "rule_duplicate",
    "rule_high_frequency",
    "rule_short_extreme",
    "rule_rating_deviation"
]].sum())


# ================================
# SAVE DATASET
# ================================

df.to_csv("../data/processed/trust_scored_dataset.csv", index=False)

print("\nDataset saved → trust_scored_dataset.csv")
print("Final shape:", df.shape)


Trust Score Statistics
count    719967.000000
mean          0.571736
std           0.122183
min           0.000000
25%           0.507931
50%           0.577273
75%           0.625000
max           0.998372
Name: trust_score, dtype: float64

Fake Label Distribution
fake_label
0    0.926647
1    0.073353
Name: proportion, dtype: float64

Confidence Distribution
label_confidence
uncertain    636370
high_real     57823
high_fake     25774
Name: count, dtype: int64

Suspicious Rule Counts
rule_duplicate            30320
rule_high_frequency        1349
rule_short_extreme       301324
rule_rating_deviation     11747
dtype: int64

Dataset saved → trust_scored_dataset.csv
Final shape: (719967, 29)


In [10]:
df[['rule_duplicate','rule_high_frequency','rule_short_extreme']].sum()

rule_duplicate          30320
rule_high_frequency      1349
rule_short_extreme     301324
dtype: int64

In [11]:
# Returns just the number (e.g., 7)
total_unique_scores = df['trust_score'].nunique()
print(f"Total unique trust scores: {total_unique_scores}")


Total unique trust scores: 67061


In [12]:
# Returns an array of the distinct score values
unique_scores = df['trust_score'].unique()
print(f"Unique trust scores assigned: {sorted(unique_scores)}")


Unique trust scores assigned: [np.float64(0.0), np.float64(0.0002463054187192004), np.float64(0.001973684210526294), np.float64(0.0024553571428571064), np.float64(0.005603448275862066), np.float64(0.008333333333333304), np.float64(0.01011904761904761), np.float64(0.01021021021021018), np.float64(0.01256038647342994), np.float64(0.013379043600562607), np.float64(0.013570804195804198), np.float64(0.013932291666666638), np.float64(0.016346153846153844), np.float64(0.016421568627450964), np.float64(0.01654411764705882), np.float64(0.018055555555555547), np.float64(0.020639534883720945), np.float64(0.020900974025974017), np.float64(0.02284964636094), np.float64(0.022854691075514855), np.float64(0.02307692307692305), np.float64(0.023106060606060616), np.float64(0.02320881226053642), np.float64(0.02385017536893652), np.float64(0.025694444444444464), np.float64(0.027584388185654007), np.float64(0.02859195402298853), np.float64(0.030098684210526333), np.float64(0.03125), np.float64(0.0318043718